In [ ]:
### look website

In [ ]:
HR Diagram with Mass lines

In [ ]:
def get_R_from_L_T(L, T):
    return np.sqrt(L / (4 * np.pi * consts.sigma_sb * T**4)).to(u.Rsun)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharex=True, sharey=True, layout='tight')

s = 10

# loop over different engines and create an HRD
for bcm, ax in zip(bcms.values(), axes):

    # plot the primary star
    mask = bcm["kstar_1"] < 13
    ax.scatter(np.log10(bcm["teff_1"][mask]), np.log10(bcm["lum_1"][mask]), c=[kstar_translator[k]["colour"] for k in bcm["kstar_1"][mask]], s=s)
    ax.plot(np.log10(bcm["teff_1"][mask]), np.log10(bcm["lum_1"][mask]), lw=0.5, color='black', zorder=-1)

    # plot the secondary star
    mask = bcm["kstar_2"] < 13
    ax.scatter(np.log10(bcm["teff_2"][mask]), np.log10(bcm["lum_2"][mask]), c=[kstar_translator[k]["colour"] for k in bcm["kstar_2"][mask]], s=s)
    ax.plot(np.log10(bcm["teff_2"][mask]), np.log10(bcm["lum_2"][mask]), lw=0.5, color='grey', zorder=-1)


ax.invert_xaxis()

for ax, label in zip(axes, ['METISSE', 'SSE']):
    ax.set(
        xlabel=r'$\log_{10}(T_{\mathrm{eff}}/\mathrm{K})$',
        ylabel=r'$\log_{10}(L/L_{\odot})$',
    )

    # add lines of constant radius using a grid of Teff and L, with the get_R_from_L_T function
    

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # create a range of temperatures and luminosities (and use Astropy units)
    T_eff = np.logspace(xlim[0], xlim[1], 1000) * u.K
    Lum = np.logspace(ylim[0], ylim[1], 1000) * u.Lsun
    
    # turn these into a grid
    L, T = np.meshgrid(Lum, T_eff)
    
    # compute the radius for this grid and convert to solar radii
    R = get_R_from_L_T(L, T)
    
    # make a contour plot of these values
    cont = ax.contour(np.log10(T.value), np.log10(L.value), np.log10(R.to(u.Rsun).value),
                        levels=[-3, -2, -1, 0, 1, 2, 3], colors="grey", zorder=1, linewidths=0.5, linestyles="dotted")
    
    # format the labels nicely with solar radii
    def fmt(x):
        if x < 0:
            return "{}".format(10**(x)) + r"$\,{\rm R_{\odot}}$"
        else:
            return "{}".format(int(np.round(10**(x)))) + r"$\,{\rm R_{\odot}}$"

    # label the lines
    ax.clabel(cont, fmt=fmt, use_clabeltext=True, rightside_up=True, fontsize=0.5*fs)

    ax.annotate(label, xy=(0.95, 0.95), xycoords='axes fraction', fontsize=fs, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', lw=0),
                weight='bold')
    
    for kstar in range(1,10):
        ax.scatter([], [], c=[kstar_translator[kstar]["colour"]], label=kstar_translator[kstar]["short"], s=30)
    ax.legend(title='Stellar Type', fontsize=0.5*fs, title_fontsize=0.6*fs, loc='lower right')